# Can we predict 30-day readmission (`readmit_30d`)?

Plan: inspect data → note that rows are repeated visits by the same person (1,500 people, ~8 visits each) → evaluate with **person-grouped** cross-validation so no person appears in both train and test → compare against baselines and a random-split (leaky) estimate.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=0
df=pd.read_csv('data.csv'); pd.read_csv('columns.csv')

## 1. Data checks

In [ ]:
print(df.shape, 'missing:', df.isna().sum().sum(), 'dup rows:', df.duplicated().sum())
print('people:', df.person_key.nunique(), '| visits/person:', df.groupby('person_key').size().describe()[['min','50%','max']].tolist())
print('readmit rate: %.3f' % df.readmit_30d.mean())
df.describe().T

In [ ]:
num=['visit_number','birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
fig,ax=plt.subplots(2,4,figsize=(16,7))
for a,col in zip(ax.flat,num+['postcode_area']):
    r=df.groupby(pd.qcut(df[col],10,duplicates='drop'),observed=True).readmit_30d.mean()
    a.plot(range(len(r)),r.values,marker='o'); a.set_title(col); a.set_xlabel('decile'); a.axhline(df.readmit_30d.mean(),ls='--',c='grey')
    a.set_ylabel('readmit rate')
plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())

## 2. Is there person-level structure?
If some people are readmitted far more often than others, a random row split would let the model memorise people. Check the between-person spread against what chance alone would give.

In [ ]:
g=df.groupby('person_key').readmit_30d.agg(['mean','size'])
p=df.readmit_30d.mean()
rng=np.random.default_rng(SEED)
sim=[pd.Series(rng.random(len(df))<p).groupby(df.person_key.values).mean().var() for _ in range(50)]
print('observed var of per-person rate: %.4f | expected under no person effect: %.4f' % (g['mean'].var(), np.mean(sim)))

## 3. Models, evaluated with person-grouped, stratified CV

In [ ]:
X=df.drop(columns=['person_key','readmit_30d']); y=df.readmit_30d.values; groups=df.person_key.values
cat=['sex','postcode_area']; numc=[c for c in X if c not in cat]
pre=ColumnTransformer([('n',StandardScaler(),numc),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
Xh=X.assign(sex=(X.sex=='M').astype(int))  # numeric copy for gradient boosting
models={
 'prevalence baseline':(DummyClassifier(strategy='prior'),X),
 'logistic regression':(make_pipeline(pre,LogisticRegression(max_iter=2000,C=0.5)),X),
 'gradient boosting':(HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,
                        categorical_features=[Xh.columns.get_loc('postcode_area')],random_state=SEED),Xh),
}
cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
oof={};rows=[]
for n,(m,D) in models.items():
    oof[n]=cross_val_predict(m,D,y,groups=groups,cv=cv,method='predict_proba')[:,1]
    rows.append(dict(model=n,AUC=roc_auc_score(y,oof[n]),AP=average_precision_score(y,oof[n]),Brier=brier_score_loss(y,oof[n])))
res=pd.DataFrame(rows).set_index('model').round(4); res

In [ ]:
# uncertainty: bootstrap over people
idx=pd.Series(np.arange(len(y))).groupby(groups).apply(list); ids=idx.index.values
def boot(pred,B=300):
    out=[]
    for _ in range(B):
        s=np.concatenate([idx[i] for i in rng.choice(ids,len(ids))]); out.append(roc_auc_score(y[s],pred[s]))
    return np.percentile(out,[2.5,97.5])
for n in ['logistic regression','gradient boosting']:
    print(n,'AUC 95%% CI (bootstrap over people): %.3f-%.3f' % tuple(boot(oof[n])))

### Leakage check: same model with a naive random-row split

In [ ]:
m,D=models['gradient boosting']
rnd=cross_val_predict(m,D,y,cv=KFold(5,shuffle=True,random_state=SEED),method='predict_proba')[:,1]
print('GB AUC random split: %.3f  vs grouped split: %.3f' % (roc_auc_score(y,rnd),roc_auc_score(y,oof['gradient boosting'])))

## 4. Discrimination, calibration, drivers

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(17,5))
for n in list(models)[1:]:
    f,t,_=roc_curve(y,oof[n]); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(y,oof[n]); ax[1].plot(rc,pr,label=n)
    fr,mp=calibration_curve(y,oof[n],n_bins=10,strategy='quantile'); ax[2].plot(mp,fr,marker='o',label=n)
ax[0].plot([0,1],[0,1],'k--');ax[0].set(title='ROC',xlabel='FPR',ylabel='TPR')
ax[1].axhline(y.mean(),c='k',ls='--');ax[1].set(title='Precision-recall',xlabel='recall',ylabel='precision')
ax[2].plot([0,1],[0,1],'k--');ax[2].set(title='Calibration',xlabel='predicted',ylabel='observed')
ax[0].legend();plt.tight_layout();plt.show()

In [ ]:
# permutation importance on held-out people (single grouped split)
tr,te=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
m,D=models['gradient boosting']; m.fit(D.iloc[tr],y[tr])
pi=permutation_importance(m,D.iloc[te],y[te],scoring='roc_auc',n_repeats=20,random_state=SEED)
imp=pd.Series(pi.importances_mean,D.columns).sort_values(); imp.plot.barh(xerr=pd.Series(pi.importances_std,D.columns)[imp.index]); plt.xlabel('AUC drop'); plt.show()
lr=models['logistic regression'][0].fit(X,y)
coef=pd.Series(lr[-1].coef_[0],lr[0].get_feature_names_out()); coef[coef.index.str.startswith('n__')].sort_values()

In [ ]:
# risk stratification: observed readmission rate by predicted-risk decile (out of fold)
d=pd.DataFrame({'p':oof['gradient boosting'],'y':y}); d['decile']=pd.qcut(d.p,10,labels=False)+1
d.groupby('decile').agg(pred=('p','mean'),obs=('y','mean'),n=('y','size')).round(3)

## 5. Conclusions (from a run with seed 0; re-run to confirm)

- **Weak but real signal.** Person-grouped 5-fold CV: logistic regression AUC ≈ 0.62 (bootstrap 95% CI ≈ 0.61–0.64), average precision 0.30 vs 0.20 prevalence. Gradient boosting is worse (AUC ≈ 0.60) and poorly calibrated, so the simple model is preferred. This is far from a reliable individual-level predictor.
- **Drivers:** longer stay, more medications and higher `lab_a` raise risk; the other features add essentially nothing (sex, postcode, `lab_b`, BMI, visit number ≈ 0).
- **Leakage trap:** readmission clusters strongly by person (variance of per-person rates ≈ 0.071 vs ≈ 0.023 expected by chance). A naive random row split inflates the gradient-boosting AUC to ≈ 0.69 vs 0.60 grouped, because the model recognises people it has seen. Any estimate must be on unseen people.
- **Limits:** no timing information, so we can't test forward-in-time performance; the person effect itself is unexplained by the available columns (it may be a latent patient factor such as chronic illness) and would likely help if prior-visit history were available for the same person at prediction time, but that must be validated temporally.